# 📸 Memento: diseño de agentes — Live coding (docente)
**Patrones de planificación y memoria de agentes** · Autora: Elena Díaz ([@HelenDiMo](https://github.com/HelenDiMo/pildora-bootcamp-memento-agentes))

> Leonard no puede crear recuerdos nuevos. Cada pocos minutos se «despierta» sin saber qué estaba haciendo.
> Un LLM funciona igual: **no recuerda nada entre llamadas**. Hoy le damos a Leonard un método (planificación) y unas notas (memoria).

| Paso | Qué construimos | Min |
|---|---|---|
| 0 | Elegir LLM | 1 |
| 1 | Leonard sin memoria | 2 |
| 2 | Herramientas: sus manos | 3 |
| 3 | Agente ReAct a mano con LangGraph | 5 |
| 4 | Memoria a corto plazo (hilos) | 4 |
| 5 | Plan-and-Execute | 4 |

*Si vas justa de tiempo, el paso 5 se puede enseñar solo con la web.*

## 0 · Elegir el LLM

In [ ]:
# ⚙️ Elige tu LLM: "groq" (clave en .env) u "ollama" (local, modelo ya descargado)
PROVEEDOR = "groq"

from memento_kit import get_llm, mostrar_traza, PROMPT_LEONARD
llm = get_llm(PROVEEDOR)
print("LLM listo:", llm.__class__.__name__)

## 1 · Leonard sin memoria
Dos llamadas seguidas al mismo LLM. ¿Recuerda la segunda lo que dijimos en la primera?

In [ ]:
r1 = llm.invoke("Me llamo Helen y estoy buscando a un tal J.G.")
print("1ª llamada →", r1.content[:300])

r2 = llm.invoke("¿Cómo me llamo y a quién busco?")
print("\n2ª llamada →", r2.content[:300])

💡 **El LLM es *stateless* (sin estado).** Cada `invoke` es un Leonard recién despertado.
Lo que llamamos «memoria» en un chat es solo que **le reenviamos el historial** en cada llamada.

## 2 · Herramientas: las manos de Leonard
Una herramienta es una función de Python con un **docstring**: el LLM lee el docstring para decidir cuándo usarla.
Las del caso están en `memento_kit.py`.

In [ ]:
from memento_kit import buscar_archivo, ver_polaroid

herramientas = [buscar_archivo, ver_polaroid]
for h in herramientas:
    print(f"🔧 {h.name}{h.args}\n   {h.description}\n")

In [ ]:
# El LLM DECIDE usar una herramienta... pero no la ejecuta él
llm_con_manos = llm.bind_tools(herramientas)
decision = llm_con_manos.invoke("¿Qué dice el informe del incidente?")
print("Texto:", repr(decision.content))
print("Llamadas a herramientas:", decision.tool_calls)

💡 El LLM solo devuelve **qué herramienta quiere usar y con qué argumentos**. Alguien tiene que ejecutarla y devolverle el resultado.
Ese «alguien» es el bucle del agente 👇

## 3 · Patrón ReAct: piensa → actúa → observa → repite
Lo construimos **a mano** con LangGraph para ver todas las piezas:

- **Estado** (`MessagesState`): la lista de mensajes que va creciendo.
- **Nodo `agente`**: llama al LLM.
- **Nodo `herramientas`**: ejecuta las herramientas que pidió el LLM.
- **Arista condicional**: si el LLM pidió herramientas → `herramientas`; si no → fin.

In [ ]:
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

def agente(estado: MessagesState):
    mensajes = [SystemMessage(PROMPT_LEONARD)] + estado["messages"]
    return {"messages": [llm_con_manos.invoke(mensajes)]}

g = StateGraph(MessagesState)
g.add_node("agente", agente)
g.add_node("herramientas", ToolNode(herramientas))
g.add_edge(START, "agente")
g.add_conditional_edges("agente", tools_condition, {"tools": "herramientas", END: END})
g.add_edge("herramientas", "agente")   # ← el bucle: observa y vuelve a pensar

react = g.compile()
print(react.get_graph().draw_mermaid())

In [ ]:
resultado = react.invoke({"messages": [("user", "¿Quién es J.G. y dónde puedo encontrarlo?")]})
mostrar_traza(resultado["messages"])

💡 Fíjate en la traza: 🔧 acción → 👁 observación → 🔧 otra acción... El agente **decide sobre la marcha** el siguiente paso.
Esto es planificación *implícita*: no hay plan escrito, solo el siguiente paso.

## 4 · Memoria a corto plazo: la conversación actual
Compilamos el **mismo grafo** con un *checkpointer*: guarda el estado de cada **hilo** (`thread_id`).
Mismo hilo = Leonard recuerda. Hilo nuevo = Leonard se despierta en blanco.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

react_mem = g.compile(checkpointer=InMemorySaver())

hilo_1 = {"configurable": {"thread_id": "polaroid-1"}}
react_mem.invoke({"messages": [("user", "Me llamo Helen. Busca qué dice el archivo sobre el coche.")]}, hilo_1)

r = react_mem.invoke({"messages": [("user", "¿Cómo me llamo y qué te pedí antes?")]}, hilo_1)
print("Hilo 1 →", r["messages"][-1].content)

In [ ]:
hilo_2 = {"configurable": {"thread_id": "polaroid-2"}}
r = react_mem.invoke({"messages": [("user", "¿Cómo me llamo y qué te pedí antes?")]}, hilo_2)
print("Hilo 2 →", r["messages"][-1].content)

In [ ]:
# ¿Qué guarda el checkpointer? Todo el historial del hilo
estado = react_mem.get_state(hilo_1)
print("Mensajes guardados en polaroid-1:", len(estado.values["messages"]))

💡 **Memoria a corto plazo = historial del hilo.** Problema: crece en cada turno y la ventana de contexto es finita.
Soluciones habituales: recortar los mensajes antiguos o resumirlos.

## 5 · Plan-and-Execute: primero el plan, luego los pasos
En vez de improvisar paso a paso, Leonard **escribe un plan completo** (como sus notas) y lo ejecuta en orden.
Usamos **Pydantic** para obligar al LLM a devolver el plan con una estructura fija.

In [ ]:
import operator
from typing import Annotated, TypedDict
from pydantic import BaseModel, Field

class Plan(BaseModel):
    pasos: list[str] = Field(description="Entre 2 y 4 pasos cortos y ordenados")

class EstadoPlan(TypedDict):
    tarea: str
    plan: list[str]
    hechos: Annotated[list[str], operator.add]   # se van ACUMULANDO
    respuesta: str

planificador = llm.with_structured_output(Plan)

def planificar(estado: EstadoPlan):
    plan = planificador.invoke(
        f"Eres Leonard. Crea un plan para: {estado['tarea']}\n"
        "Herramientas: buscar_archivo(tema: informe|motel|coche|taller) y ver_polaroid(nombre: teddy|natalie|motel)."
    )
    return {"plan": plan.pasos}

def ejecutar_paso(estado: EstadoPlan):
    paso, *resto = estado["plan"]
    ya_sabes = "\n".join(estado["hechos"]) or "nada todavía"
    r = react.invoke({"messages": [("user", f"Lo que ya sabes:\n{ya_sabes}\n\nHaz SOLO este paso: {paso}")]})
    return {"plan": resto, "hechos": [f"{paso} → {r['messages'][-1].content}"]}

def quedan_pasos(estado: EstadoPlan):
    return "ejecutar_paso" if estado["plan"] else "informe"

def informe(estado: EstadoPlan):
    r = llm.invoke(f"Tarea: {estado['tarea']}\nHallazgos:\n" + "\n".join(estado["hechos"])
                   + "\nEscribe la conclusión final en 3 frases.")
    return {"respuesta": r.content}

gp = StateGraph(EstadoPlan)
gp.add_node("planificar", planificar)
gp.add_node("ejecutar_paso", ejecutar_paso)
gp.add_node("informe", informe)
gp.add_edge(START, "planificar")
gp.add_conditional_edges("planificar", quedan_pasos)
gp.add_conditional_edges("ejecutar_paso", quedan_pasos)
gp.add_edge("informe", END)
plan_y_ejecuta = gp.compile()

In [ ]:
entrada = {"tarea": "Averiguar dónde está ahora J.G.", "plan": [], "hechos": [], "respuesta": ""}

for evento in plan_y_ejecuta.stream(entrada, stream_mode="updates"):
    for nodo, cambio in evento.items():
        if nodo == "planificar":
            print("📝 PLAN:"); [print(f"   {i}. {p}") for i, p in enumerate(cambio["plan"], 1)]
        elif nodo == "ejecutar_paso":
            print("✅", cambio["hechos"][0][:220])
        elif nodo == "informe":
            print("\n📸 CONCLUSIÓN:", cambio["respuesta"])

💡 **ReAct vs Plan-and-Execute**

| | ReAct | Plan-and-Execute |
|---|---|---|
| Plan | Implícito, paso a paso | Explícito, al principio |
| Ventaja | Flexible ante sorpresas | Ordenado, más barato si el plan es bueno |
| Riesgo | Se puede perder en bucles | Plan rígido si algo cambia → **replanificar** |

**Variante:** añadir un nodo `replanificar` después de cada paso que reescriba el plan con lo aprendido.

➡️ **Ahora te toca:** abre `02_reto_individual_TODO.ipynb`.